In [1]:
# Comparación de Modelos Predictivos y Entrenamiento de modelos
## Random Forest, Gradient Boosting, XGBoost

In [2]:
import sys
!{sys.executable} -m pip install optuna
!{sys.executable} -m pip install xgboost

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   ---------------------------------------- 0/4 [Mako]
   --------------------------------------

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
import warnings

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

In [3]:
#Cargar dataset
df = pd.read_excel(
    "../data/raw/dataset_sintetico_demanda_lima.xlsx"
)
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha")

In [4]:
#Ingeniería de características
# Features
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['dia'] = df['fecha'].dt.day

for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['demanda_real'].shift(lag)

df['media_movil_7'] = df['demanda_real'].rolling(7).mean()
df = df.dropna().reset_index(drop=True)

In [9]:
#Variables predictoras
feature_cols = [
    'dia_semana',
    'mes',
    'dia',
    'es_festivo',
    'temperatura_promedio',
    'precipitacion_mm',
    'lag_1',
    'lag_2',
    'lag_3',
    'lag_7',
    'media_movil_7'
]

In [11]:
# Dividir datos (80% entrenamiento, 20% prueba)
X = df[feature_cols]
y = df['demanda_real']
split = int(len(df) * 0.8)
X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(f"Datos: {len(df)} días")
print(f"Train: {len(X_train)} días, Test: {len(X_test)} días")
print(f"Features: {len(feature_cols)}")


Datos: 723 días
Train: 578 días, Test: 145 días
Features: 11


In [12]:
# ============================================
# FUNCIÓN PARA CALCULAR LAS 5 MÉTRICAS
# ============================================
def calcular_metricas(y_real, y_pred, n_features):
    mae = mean_absolute_error(
        y_real,
        y_pred
    )
    rmse = np.sqrt(
        mean_squared_error(
            y_real,
            y_pred
        )
    )
    mape = (
        np.mean(
            np.abs(
                (y_real - y_pred)
                / y_real
            )
        ) * 100
    )
    r2 = r2_score(
        y_real,
        y_pred
    )
    n = len(y_real)
    r2_adj = (
        1 -
        (
            (1-r2)*(n-1)
            /
            (n-n_features-1)
        )
    )
    return {
        "MAE": round(mae,2),
        "RMSE": round(rmse,2),
        "MAPE": round(mape,2),
        "R2": round(r2,4),
        "R2_Ajustado": round(r2_adj,4)
    }

In [13]:
#OPTUNA PARA RANDOM FOREST
def objective_rf(trial):
    params = {
        "n_estimators":
        trial.suggest_int(
            "n_estimators",
            100,
            500
        ),
        "max_depth":
        trial.suggest_int(
            "max_depth",
            3,
            20
        ),
        "min_samples_split":
        trial.suggest_int(
            "min_samples_split",
            2,
            10
        )
    }
    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )
    model.fit(
        X_train,
        y_train
    )
    pred = model.predict(X_test)
    return mean_absolute_error(
        y_test,
        pred
    )

In [14]:
#Ejecución para Random Forest
study_rf = optuna.create_study(
    direction="minimize"
)
study_rf.optimize(
    objective_rf,
    n_trials=30
)
study_rf.best_params

[I 2026-06-18 01:18:23,343] A new study created in memory with name: no-name-5ae0de32-4048-4cbb-83ff-62088d4302e1
[I 2026-06-18 01:18:27,058] Trial 0 finished with value: 12.273338706044642 and parameters: {'n_estimators': 446, 'max_depth': 7, 'min_samples_split': 5}. Best is trial 0 with value: 12.273338706044642.
[I 2026-06-18 01:18:27,806] Trial 1 finished with value: 12.738726051769946 and parameters: {'n_estimators': 126, 'max_depth': 4, 'min_samples_split': 4}. Best is trial 0 with value: 12.273338706044642.
[I 2026-06-18 01:18:31,825] Trial 2 finished with value: 12.25889379860654 and parameters: {'n_estimators': 491, 'max_depth': 18, 'min_samples_split': 2}. Best is trial 2 with value: 12.25889379860654.
[I 2026-06-18 01:18:33,860] Trial 3 finished with value: 12.340190956871993 and parameters: {'n_estimators': 252, 'max_depth': 19, 'min_samples_split': 2}. Best is trial 2 with value: 12.25889379860654.
[I 2026-06-18 01:18:34,738] Trial 4 finished with value: 12.41882731457827 

{'n_estimators': 484, 'max_depth': 15, 'min_samples_split': 10}

In [15]:
#OPTUNA PARA GRADIENT BOOSTING
def objective_gb(trial):
    params = {
        "n_estimators":
        trial.suggest_int(
            "n_estimators",
            100,
            500
        ),
        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            0.01,
            0.3
        ),
        "max_depth":
        trial.suggest_int(
            "max_depth",
            2,
            8
        )
    }
    model = GradientBoostingRegressor(
        **params,
        random_state=42
    )
    model.fit(
        X_train,
        y_train
    )
    pred = model.predict(X_test)
    return mean_absolute_error(
        y_test,
        pred
    )

In [16]:
#Ejecución para Gradient Boosting
study_gb = optuna.create_study(
    direction="minimize"
)
study_gb.optimize(
    objective_gb,
    n_trials=30
)
study_gb.best_params

[I 2026-06-18 01:19:48,432] A new study created in memory with name: no-name-389722cf-c393-458f-bbe5-27d8366cf8e5
[I 2026-06-18 01:19:50,950] Trial 0 finished with value: 12.684719346589814 and parameters: {'n_estimators': 217, 'learning_rate': 0.2540901229696883, 'max_depth': 6}. Best is trial 0 with value: 12.684719346589814.
[I 2026-06-18 01:19:52,366] Trial 1 finished with value: 12.533839286239374 and parameters: {'n_estimators': 245, 'learning_rate': 0.1485701755466148, 'max_depth': 3}. Best is trial 1 with value: 12.533839286239374.
[I 2026-06-18 01:19:54,055] Trial 2 finished with value: 13.039262466688657 and parameters: {'n_estimators': 289, 'learning_rate': 0.24659631508335128, 'max_depth': 3}. Best is trial 1 with value: 12.533839286239374.
[I 2026-06-18 01:19:59,382] Trial 3 finished with value: 12.754096723376607 and parameters: {'n_estimators': 453, 'learning_rate': 0.05517129324121987, 'max_depth': 7}. Best is trial 1 with value: 12.533839286239374.
[I 2026-06-18 01:20:

{'n_estimators': 244, 'learning_rate': 0.10392047474434872, 'max_depth': 2}

In [17]:
#OPTUNA PARA XGBOOST
def objective_xgb(trial):
    params = {
        "n_estimators":
        trial.suggest_int(
            "n_estimators",
            100,
            500
        ),
        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            0.01,
            0.3
        ),
        "max_depth":
        trial.suggest_int(
            "max_depth",
            3,
            10
        ),
        "subsample":
        trial.suggest_float(
            "subsample",
            0.6,
            1.0
        )
    }
    model = XGBRegressor(
        **params,
        random_state=42,
        verbosity=0
    )
    model.fit(
        X_train,
        y_train
    )
    pred = model.predict(
        X_test
    )
    return mean_absolute_error(
        y_test,
        pred
    )

In [18]:
#Ejecución para XGBoost
study_xgb = optuna.create_study(
    direction="minimize"
)
study_xgb.optimize(
    objective_xgb,
    n_trials=30
)
study_xgb.best_params

[I 2026-06-18 01:20:47,265] A new study created in memory with name: no-name-0cbc29ff-5b34-488e-9a7d-17d6e34a82a2
[I 2026-06-18 01:20:49,948] Trial 0 finished with value: 13.854723930358887 and parameters: {'n_estimators': 153, 'learning_rate': 0.2505885247546846, 'max_depth': 9, 'subsample': 0.9297847356662429}. Best is trial 0 with value: 13.854723930358887.
[I 2026-06-18 01:20:50,798] Trial 1 finished with value: 12.053059577941895 and parameters: {'n_estimators': 330, 'learning_rate': 0.1420292165138768, 'max_depth': 3, 'subsample': 0.7055601824192398}. Best is trial 1 with value: 12.053059577941895.
[I 2026-06-18 01:20:51,207] Trial 2 finished with value: 12.179044723510742 and parameters: {'n_estimators': 267, 'learning_rate': 0.1799373415041483, 'max_depth': 3, 'subsample': 0.8248769199173474}. Best is trial 1 with value: 12.053059577941895.
[I 2026-06-18 01:20:52,964] Trial 3 finished with value: 12.57468032836914 and parameters: {'n_estimators': 190, 'learning_rate': 0.0438983

{'n_estimators': 284,
 'learning_rate': 0.056492383763733646,
 'max_depth': 4,
 'subsample': 0.8327732172809209}

In [19]:
#ENTREAMIENTO FINAL PARA LOS MODELOS
#1. Random Forest
rf = RandomForestRegressor(
    **study_rf.best_params,
    random_state=42
)
#2. Gradient Boosting
gb = GradientBoostingRegressor(
    **study_gb.best_params,
    random_state=42
)
#3. XGBoost
xgb = XGBRegressor(
    **study_xgb.best_params,
    random_state=42,
    verbosity=0
)

In [20]:
#Ejecución de entrenamiento de los modelos
rf.fit(X_train,y_train)
gb.fit(X_train,y_train)
xgb.fit(X_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [21]:
#Predicciones
y_pred_rf = rf.predict(X_test)
y_pred_gb = gb.predict(X_test)
y_pred_xgb = xgb.predict(X_test)

In [22]:
#Métricas
metrics_rf = calcular_metricas(
    y_test,
    y_pred_rf,
    len(feature_cols)
)
metrics_gb = calcular_metricas(
    y_test,
    y_pred_gb,
    len(feature_cols)
)
metrics_xgb = calcular_metricas(
    y_test,
    y_pred_xgb,
    len(feature_cols)
)

In [23]:
#TABLA FINAL
resultados = pd.DataFrame([
    {
        "Modelo":"Random Forest",
        **metrics_rf
    },
    {
        "Modelo":"Gradient Boosting",
        **metrics_gb
    },
    {
        "Modelo":"XGBoost",
        **metrics_xgb
    }
])
resultados = resultados.sort_values(
    "MAE"
)
resultados

,Modelo,MAE,RMSE,MAPE,R2,R2_Ajustado
2,XGBoost,11.63,15.31,5.71,0.7872,0.7695
1,Gradient Boosting,11.67,15.05,5.77,0.7945,0.7775
0,Random Forest,12.04,15.56,6.00,0.7802,0.7621


In [24]:
#Guardar CSV
import os
os.makedirs(
    "../reports/tables",
    exist_ok=True
)
resultados.to_csv(
    "../reports/tables/model_comparison_full.csv",
    index=False
)

In [25]:
#Guardar modelos
os.makedirs(
    "../data/processed",
    exist_ok=True
)
joblib.dump(
    rf,
    "../data/processed/random_forest.pkl"
)
joblib.dump(
    gb,
    "../data/processed/gradient_boosting.pkl"
)
joblib.dump(
    xgb,
    "../data/processed/xgboost.pkl"
)

['../data/processed/xgboost.pkl']